# Анализ развития игровой исндустрии с 2000 по 2013 год

- Автор:Малявина Дарья
- Дата:29.01.2026


### Цели и задачи проекта

Цель проекта: изучить развитие игровой индустрии с 2000 по 2013 год
    
Задачи:
- Познакомится с данными
- Проверить корректность данных
- Провести предобработку данных



### Описание данных

Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
    
- **Name** — название игры.
- **Platform** — название платформы.
- **Year of Release** — год выпуска игры.
- **Genre** — жанр игры.
- **NA sales** — продажи в Северной Америке (в миллионах проданных копий).
- **EU sales** — продажи в Европе (в миллионах проданных копий).
- **JP sales** — продажи в Японии (в миллионах проданных копий).
- **Other sales** — продажи в других странах (в миллионах проданных копий).
- **Critic Score** — оценка критиков (от 0 до 100).
- **User Score** — оценка пользователей (от 0 до 10).
- **Rating** — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Проверка ошибок в данных и их предобработка
3. Фильтрация данных 
4. Категоризация данных 
5. Выводы

## 1. Загрузка данных и знакомство с ними

Подключаем библиотеку pandas и выгружаем датасет

In [1]:
import pandas as pd

In [2]:
games = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

Выводим первые 5 строк датасета



In [3]:
games.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Проверяем информацию о структуре датасета


In [4]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Датафрейм состоит из 16956 строк и 11 столбцов. В 6 столбцах есть пропуски (`'Name', 'Year of Release', 'Genre', 'Critic Score', 'User Score', 'Rating'`) и в 4 столбцах используются неподходящие типы данных (`'Year of Release', 'EU sales', 'JP sales', 'User Score'`).




---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма


Выводим названия всех столбцоа датасета


In [5]:
games.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

Для удобства работы с данными приведем все столбца к стилю snake case


Меняем названия столбцов с помощью метода rename()

In [6]:
games = games.rename(columns={'Name':'name', 'Platform':'platform','Year of Release':'year_of_release', 'Genre': 'genre', 'NA sales':'na_sales', 'EU sales':'eu_sales', 'JP sales': 'jp_sales', 'Other sales': 'other_sales', 'Critic Score':'critic_score', 'User Score': 'user_score', 'Rating': 'rating'})
games.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### 2.2. Типы данных

В 4 столбцах используются неподходящие типы данных (`'Year of Release', 'EU sales', 'JP sales', 'User Score'`). 

- `'Year of Release'` - год логичнее преобразовать к типу int64, но в столбце есть пропуски
- `'EU sales', 'JP sales'` - продажи надо преобразовать к float64, пропусков в столбцах нет
- `'User Score'` - оценки стоит перобразовать к float64


In [7]:
#Преобразуем данные столбцов 'EU sales', 'JP sales', 'User Score', используя метод to_numeric()
#Метод позволяет заменить строковые значения на пропуски
#С помощью параметра downcast подбираем подходящий битовый разряд
games['eu_sales'] = pd.to_numeric(games['eu_sales'], errors = 'coerce', downcast = 'float')
games['jp_sales'] = pd.to_numeric(games['jp_sales'], errors = 'coerce', downcast = 'float')
games['user_score'] = pd.to_numeric(games['user_score'], errors = 'coerce', downcast = 'float')
#Проверяем результат изменения типа данных
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  object 
dtypes: float32(3), float64(4), object(4)
memory usage: 1.2+ MB


### 2.3. Наличие пропусков в данных

Проверяем количество пропусков в абсолютных значениях


In [8]:
games.isna().sum().sort_values(ascending=False)

user_score         9268
critic_score       8714
rating             6871
year_of_release     275
eu_sales              6
jp_sales              4
name                  2
genre                 2
platform              0
na_sales              0
other_sales           0
dtype: int64

Проверяем количество пропусков в отностительных значениях


In [9]:
games.isna().mean().sort_values(ascending=False)

user_score         0.546591
critic_score       0.513918
rating             0.405225
year_of_release    0.016218
eu_sales           0.000354
jp_sales           0.000236
name               0.000118
genre              0.000118
platform           0.000000
na_sales           0.000000
other_sales        0.000000
dtype: float64

В 6 столбцах есть пропуски (`'Name', 'Year of Release', 'Genre', 'Critic Score', 'User Score', 'Rating'`). 

- В столбцах `'Critic Score', 'User Score'` около 50% пропусков, эти пропуски нельзя удалить (от их удаления может исказиться результат анализа) или заменить на какое-то среднее значение. Я думаю лучшим решением будет оставить их. Такое количество пропусков может быть связано с тем, что некоторые платформы не предоставляют информацию о оценке критиков и пользователей на фильмы (ниже приведен код, который показывает, что на некоторые платформы вообще не предоставлены оценки):


Проверка средней оценки критиков по платфорам

In [10]:
games.groupby(['platform'])['critic_score'].mean().head()

platform
2600          NaN
3DO           NaN
3DS     67.035503
DC      87.357143
DS      63.813536
Name: critic_score, dtype: float64

Проверка средней оценки пользователей по платфорам

In [11]:
games.groupby(['platform'])['user_score'].mean().head()

platform
2600         NaN
3DO          NaN
3DS     6.833143
DC      8.528571
DS      7.029310
Name: user_score, dtype: float32

- В столбцах `'name', 'genre',  'eu_sales', 'jp_sales'` меньше 1% пропусков. Эти пропуски вероятнее всего относятся к типу MCAR, так как сложно их как-то объяснить (ниже можно увидеть строки с этими пропусками). Так как пропусков мало, то их можно удалить.

Проверка строк с пропусками в названии игр

In [12]:
n = games[(games['name'].isna() == True)]
n

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


Проверка строк с пропусками в европейских продажах


In [13]:
a = games[(games['eu_sales'].isna() == True)]
a 

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
446,Rhythm Heaven,DS,2008.0,Misc,0.55,NaN,1.93,0.13,83.0,9.0,E
802,Dead Rising,X360,2006.0,Action,1.16,NaN,0.08,0.20,85.0,7.6,M
1131,Prince of Persia: Warrior Within,PS2,2004.0,Action,0.54,NaN,0.00,0.22,83.0,8.5,M
1132,Far Cry 4,XOne,2014.0,Shooter,0.80,NaN,0.01,0.14,82.0,7.5,M
1394,Sonic Advance 3,GBA,2004.0,Platform,0.74,NaN,0.08,0.06,79.0,8.4,E
1612,Ratatouille,DS,2007.0,Action,0.49,NaN,0.00,0.14,NaN,NaN,NaN


Проверка строк с пропусками в японских продажах

In [14]:
b = games[(games['jp_sales'].isna() == True)]
b

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
467,Saints Row 2,X360,2008.0,Action,1.94,0.79,NaN,0.28,81.0,8.1,M
819,UFC 2009 Undisputed,X360,2009.0,Fighting,1.48,0.39,NaN,0.19,83.0,7.9,T
1379,Hello Kitty Party,DS,2007.0,Misc,0.78,0.51,NaN,0.12,NaN,NaN,E
4732,Castlevania: The Dracula X Chronicles,PSP,2007.0,Platform,0.22,0.09,NaN,0.07,80.0,7.8,T


- В столбце `'rating'` около 40% пропусков, поэтому удалить их не получится (данные исказяться), но данный столбец никак не участвует в дальнейшей работе, поэтому пропуски в нем можно оставить как есть. Можно заметить, что пропуски в рейтенге также как и оценки связаны с платфорами. На платформах, на которых отстутствует информация об оценках, не предоставляется и информацию о рейтенге.


Проверка количества пометок с рейтингом по каждой платформе

In [15]:
games.groupby(['platform'])['rating'].count().head()

platform
2600       0
3DO        0
3DS      232
DC        14
DS      1285
Name: rating, dtype: int64

- В столбце `'year_of_release'` чуть больше 1% пропусков, что не сильно повлияет на дальнейшую работу. Но так как я хотела бы перевести данный столбец к типу данных int64 для удобства, я заменю пропуски на числовой индикатор '0'. Я думаю пропуски в этом столбце относятся к типу MCAR, и для их появления нет точного объяснения.

Удалаем строки с пропусками в столбцах 'name', 'genre',  'eu_sales', 'jp_sales'


In [16]:
games = games.dropna(subset=['name', 'genre',  'eu_sales', 'jp_sales'])

Заменяем пропуски на индикатор 0 в столбце 'year_of_release'


In [17]:
games.loc[:, 'year_of_release'] = games['year_of_release'].fillna(0)

Изменяем тип данных столбца 'year_of_release'


In [18]:
games['year_of_release'] = pd.to_numeric(games['year_of_release'], downcast = 'integer')

Проверяем информацию о структуре датасета


In [19]:
games.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16944 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16944 non-null  object 
 1   platform         16944 non-null  object 
 2   year_of_release  16944 non-null  int16  
 3   genre            16944 non-null  object 
 4   na_sales         16944 non-null  float64
 5   eu_sales         16944 non-null  float32
 6   jp_sales         16944 non-null  float32
 7   other_sales      16944 non-null  float64
 8   critic_score     8234 non-null   float64
 9   user_score       7680 non-null   float32
 10  rating           10076 non-null  object 
dtypes: float32(3), float64(3), int16(1), object(4)
memory usage: 1.3+ MB


### 2.4. Явные и неявные дубликаты в данных

Проверяем уникальные значения в названии жанра

In [20]:
uniq_genre = games['genre'].unique()
uniq_genre

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

В столбце есть неявные дубликаты. Все названия повторяются в верхнем регистре. Чтобы избавиться от дубликатов все значения переведу к нижнему регистру.



In [21]:
#Переводим названия в нижний регистр
games['genre'] = games['genre'].str.lower()
#Проверяем изменения
uniq_genre = games['genre'].unique()
uniq_genre

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

Проверяем уникальные значения в названии платформ

In [22]:
uniq_platform = games['platform'].unique()
uniq_platform

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

Дубликатов нет

Проверяем уникальные значения в названии рейтинга


In [23]:
uniq_rating = games['rating'].unique()
uniq_rating

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

Дубликатов нет, но появляется странное название рейтинга, которого нет в списке - `'K-A'`

Проверяем уникальные значения в столбце с годом


In [24]:
uniq_year_of_release = games['year_of_release'].unique()
uniq_year_of_release

array([2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010,
       2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014,
       1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000,    0, 1995,
       1991, 1981, 1987, 1980, 1983], dtype=int16)

Дубликатов нет

Проверяем наличие явных дубоикатов 



In [25]:
dup = games[games.duplicated()]
dup

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
268,Batman: Arkham Asylum,PS3,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
368,James Bond 007: Agent Under Fire,PS2,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
717,God of War: Ascension,PS3,2013,action,1.23,0.63,0.04,0.35,80.0,7.5,M
823,Wipeout: The Game,Wii,2009,misc,1.94,0.00,0.00,0.12,NaN,NaN,NaN
848,Rayman Raving Rabbids: TV Party,Wii,2008,misc,0.72,1.08,0.00,0.23,73.0,7.7,E10+
...,...,...,...,...,...,...,...,...,...,...,...
16671,Fullmetal Alchemist: Prince of the Dawn,Wii,2009,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16753,Routes PE,PS2,2007,adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16799,Transformers: Prime,Wii,2012,action,0.00,0.01,0.00,0.00,NaN,NaN,NaN
16912,Metal Gear Solid V: The Definitive Experience,XOne,2016,action,0.01,0.00,0.00,0.00,NaN,NaN,M


Удаляем явные дубликаты


In [26]:
games_new = games.drop_duplicates()

Проверяем наличие явных дубликатов после удаления

In [27]:
dup_new = games_new[games_new.duplicated()]
dup_new

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating


При проверке было найдено и удалено около 240 явных дубликатов

Проверяем количество удаленных строк в абсолютном значении


In [28]:
ab = 16956 - games_new.shape[0]
ab

253

Проверяем количество удаленных строк в относительном значении

In [29]:
ot = ab/16956
ot

0.014920971927341355

Основные результаты предобработки данных:

- Названия столбцов приведены к стилю snake case
- Столбец `'Year of Release'` преобразован к типу int16, проауски в нем заменены на индикатор '0'
- Столбцы `'EU sales', 'JP sales'` преобразованы к типу float32
- Столбец `'User Score'` перобразован к float32
- В столбцах `'Critic Score', 'User Score'` пропуски были оставлены без изменений
- В столбцах `'name', 'genre',  'eu_sales', 'jp_sales'` пропуски удалены 
- В столбце `'rating'` пропуски были оставлены бкз изменения
- В столбце `'genre'` неявные дубликаты были обработаны переводом названий всех значений к нижнему регистру
- Было удалено около 240 явных дубликатов


---

## 3. Фильтрация данных

In [30]:
#Создаем срез данных с помощью индексации и условных операторов
df_actual = games_new[(games_new['year_of_release'] >= 2000) & (games_new['year_of_release'] <= 2013)]
#Проверяем новый датафрейм
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006,sports,41.36,28.959999,3.77,8.45,76.0,8.0,E
2,Mario Kart Wii,Wii,2008,racing,15.68,12.760000,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009,sports,15.61,10.930000,3.28,2.95,80.0,8.0,E
6,New Super Mario Bros.,DS,2006,platform,11.28,9.140000,6.50,2.88,89.0,8.5,E
7,Wii Play,Wii,2006,misc,13.96,9.180000,2.93,2.84,58.0,6.6,E
...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003,shooter,0.01,0.000000,0.00,0.00,NaN,NaN,T
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002,platform,0.01,0.000000,0.00,0.00,NaN,NaN,NaN
16950,SCORE International Baja 1000: The Official Game,PS2,2008,racing,0.00,0.000000,0.00,0.00,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006,sports,0.00,0.010000,0.00,0.00,NaN,NaN,NaN


---

## 4. Категоризация данных
    

In [31]:
df_actual['user_score_group'] = pd.cut(df_actual['user_score'], [0,3,8,10], labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], right=False)
df_actual

C:\Users\OC\AppData\Local\Temp\ipykernel_62976\3460962922.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actual['user_score_group'] = pd.cut(df_actual['user_score'], [0,3,8,10], labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], right=False)


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_group
0,Wii Sports,Wii,2006,sports,41.36,28.959999,3.77,8.45,76.0,8.0,E,высокая оценка
2,Mario Kart Wii,Wii,2008,racing,15.68,12.760000,3.79,3.29,82.0,8.3,E,высокая оценка
3,Wii Sports Resort,Wii,2009,sports,15.61,10.930000,3.28,2.95,80.0,8.0,E,высокая оценка
6,New Super Mario Bros.,DS,2006,platform,11.28,9.140000,6.50,2.88,89.0,8.5,E,высокая оценка
7,Wii Play,Wii,2006,misc,13.96,9.180000,2.93,2.84,58.0,6.6,E,средняя оценка
...,...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003,shooter,0.01,0.000000,0.00,0.00,NaN,NaN,T,NaN
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002,platform,0.01,0.000000,0.00,0.00,NaN,NaN,NaN,NaN
16950,SCORE International Baja 1000: The Official Game,PS2,2008,racing,0.00,0.000000,0.00,0.00,NaN,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006,sports,0.00,0.010000,0.00,0.00,NaN,NaN,NaN,NaN


In [32]:
df_actual['critic_score_group'] = pd.cut(df_actual['critic_score'], [0,30,80,100], labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], right=False)
df_actual

C:\Users\OC\AppData\Local\Temp\ipykernel_62976\2646278331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actual['critic_score_group'] = pd.cut(df_actual['critic_score'], [0,30,80,100], labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], right=False)


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_group,critic_score_group
0,Wii Sports,Wii,2006,sports,41.36,28.959999,3.77,8.45,76.0,8.0,E,высокая оценка,средняя оценка
2,Mario Kart Wii,Wii,2008,racing,15.68,12.760000,3.79,3.29,82.0,8.3,E,высокая оценка,высокая оценка
3,Wii Sports Resort,Wii,2009,sports,15.61,10.930000,3.28,2.95,80.0,8.0,E,высокая оценка,высокая оценка
6,New Super Mario Bros.,DS,2006,platform,11.28,9.140000,6.50,2.88,89.0,8.5,E,высокая оценка,высокая оценка
7,Wii Play,Wii,2006,misc,13.96,9.180000,2.93,2.84,58.0,6.6,E,средняя оценка,средняя оценка
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003,shooter,0.01,0.000000,0.00,0.00,NaN,NaN,T,NaN,NaN
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002,platform,0.01,0.000000,0.00,0.00,NaN,NaN,NaN,NaN,NaN
16950,SCORE International Baja 1000: The Official Game,PS2,2008,racing,0.00,0.000000,0.00,0.00,NaN,NaN,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006,sports,0.00,0.010000,0.00,0.00,NaN,NaN,NaN,NaN,NaN


Проверяем количество игр по категориям пользовательских оценок


In [33]:
group_user = df_actual.groupby('user_score_group')['name'].count()
group_user.sort_values(ascending=False)

C:\Users\OC\AppData\Local\Temp\ipykernel_62976\656553902.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_user = df_actual.groupby('user_score_group')['name'].count()


user_score_group
средняя оценка    4078
высокая оценка    2282
низкая оценка      116
Name: name, dtype: int64

Проверяем количество игр по категориям оценок критиков

In [34]:
group_critic = df_actual.groupby('critic_score_group')['name'].count()
group_critic.sort_values(ascending=False)

C:\Users\OC\AppData\Local\Temp\ipykernel_62976\306027658.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_critic = df_actual.groupby('critic_score_group')['name'].count()


critic_score_group
средняя оценка    5421
высокая оценка    1686
низкая оценка       55
Name: name, dtype: int64

Как среди пользователей, так и среди критиков самая популярная оценка игр - средняя. Но из-за пропусков данных с оценками, часть данных не была категорезирована



In [35]:
#Считаем количество выпущенных игр для каждой платформы
sum_name = df_actual.groupby('platform')['name'].count()
#Сортируем данные по количеству игр
top_platform = sum_name.sort_values(ascending=False)
top_platform.head(7)

platform
PS2     2126
DS      2117
Wii     1275
PSP     1179
X360    1118
PS3     1087
GBA      810
Name: name, dtype: int64

---

## 5. Итоговый вывод



В ходе работы датафрейм `games_new был` отфильтрован по году выпуска игр и создан срез данных о играх, выпущенных с 2000 по 2013 год. В срез были добавлены новые столбцы с категоризацией данных об оценках игр критиками и пользователями: `user_score_group, critic_score_group`.

Основные выводы после работы с данными:

- С 2000 по 2013 год было выпущено 12772 игр, что составляет большую часть от всех выпущенных игр за период с 1980 по 2016

- Как среди пользователей, так и среди критиков самая популярная оценка игр - средняя, но пользователи чаще критиков оценивают игры низко или высоко
- Больше всего игр за период 2000 - 2013 выпустила платформа PS2 

